# 02 · LlamaIndex：专注数据接入的 RAG 框架

LangChain 是通用编排框架，而 LlamaIndex 专为「把私有数据喂给 LLM」而生——文档加载、切块、索引、查询都有开箱即用的最佳实践。

**你将学到：**

| 章节 | 内容 |
|------|------|
| 第一章 | LlamaIndex vs LangChain：定位与适用场景 |
| 第二章 | 文档加载：SimpleDirectoryReader |
| 第三章 | 索引构建：VectorStoreIndex vs SummaryIndex |
| 第四章 | 基础查询引擎 |
| 第五章 | 高级检索：SubQuestionQueryEngine |
| 第六章 | 多文档问答实战 |
| 小结 | 何时选 LlamaIndex |


## 第一章：为什么需要 LlamaIndex？

LangChain 和 LlamaIndex 并非竞争关系，而是不同层次的工具：

```mermaid
flowchart LR
    subgraph LC["LangChain — 通用编排"]
        L1[链式调用] & L2[Agent 决策] & L3[记忆管理]
    end
    subgraph LI["LlamaIndex — 专注数据"]
        I1[文档加载] --> I2[切块索引]
        I2 --> I3[高级检索]
        I3 --> I4[子问题分解]
    end
    LC -->|需要数据接入时| LI
```

**选型建议：**
- 需要多文档问答、复杂检索 → LlamaIndex
- 需要 Agent、工具调用、多步推理 → LangChain
- 两者可以组合：LlamaIndex 做检索，LangChain 做编排


In [ ]:
# 安装依赖
# pip install llama-index llama-index-embeddings-huggingface llama-index-llms-anthropic python-dotenv

import os
from dotenv import load_dotenv
load_dotenv()

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.anthropic import Anthropic

# 使用本地 HuggingFace 模型做 Embedding（无需 GPU/API Key）
Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')
# LLM 使用 Claude
Settings.llm = Anthropic(model='claude-sonnet-4-6', api_key=os.getenv('ANTHROPIC_API_KEY'))
print('配置完成')


## 第二章：文档加载

`SimpleDirectoryReader` 是 LlamaIndex 最核心的加载器，支持 PDF、Markdown、HTML、Word 等格式，自动递归遍历目录。


In [ ]:
from llama_index.core import SimpleDirectoryReader
import tempfile, os

# 创建演示文档
demo_dir = tempfile.mkdtemp()
docs_content = [
    ('climate.txt', 'Global warming is caused by greenhouse gases. CO2 levels have risen 50% since pre-industrial times. The Paris Agreement targets limiting warming to 1.5°C.'),
    ('ai_history.txt', 'Artificial intelligence began in 1956 at the Dartmouth Conference. The deep learning revolution started around 2012 with AlexNet winning ImageNet.'),
    ('health.txt', 'Regular exercise reduces cardiovascular disease risk by 35%. Sleep deprivation impairs cognitive function. Mediterranean diet is linked to longevity.'),
]
for fname, content in docs_content:
    with open(os.path.join(demo_dir, fname), 'w') as f:
        f.write(content)

# 加载文档
documents = SimpleDirectoryReader(demo_dir).load_data()
print(f'加载了 {len(documents)} 个文档')
print(f'\n第一个文档:  ID = {documents[0].doc_id}')
print(f'元数据: {documents[0].metadata}')
print(f'内容前 80 字: {documents[0].text[:80]}...')


## 第三章：索引构建

LlamaIndex 提供两种核心索引类型，适用场景不同：

```mermaid
flowchart LR
    subgraph VS["VectorStoreIndex"]
        V1[文档切块] --> V2[向量化]
        V2 --> V3[(向量数据库)]
        V3 -->|余弦相似度| V4[Top-K 检索]
    end
    subgraph SI["SummaryIndex"]
        S1[完整文档] --> S2[顺序存储]
        S2 -->|全部遍历| S3[摘要合并]
    end
    VS -->|适合：精确问答| USE1[问题 → 具体片段]
    SI -->|适合：全文摘要| USE2[文档 → 整体概述]
```


In [ ]:
from llama_index.core import VectorStoreIndex, SummaryIndex

# VectorStoreIndex：语义检索，适合精确问答
vector_index = VectorStoreIndex.from_documents(documents)
print('VectorStoreIndex 构建完成')
print(f'  存储了 {len(vector_index.docstore.docs)} 个节点')

# SummaryIndex：顺序存储，适合全文摘要
summary_index = SummaryIndex.from_documents(documents)
print('\nSummaryIndex 构建完成')
print(f'  存储了 {len(summary_index.docstore.docs)} 个节点')


## 第四章：基础查询引擎

从索引创建 `query_engine`，然后直接提问：

In [ ]:
# 基础查询引擎
query_engine = vector_index.as_query_engine(similarity_top_k=2)

response = query_engine.query('What are the main causes of global warming?')
print('回答:', response)
print()
print('来源节点:')
for node in response.source_nodes:
    print(f'  - [{node.metadata.get("file_name", "unknown")}] 相似度={node.score:.3f}')
    print(f'    {node.text[:100]}...')


## 第五章：高级检索——子问题查询

`SubQuestionQueryEngine` 把复杂问题自动拆成多个子问题，分别检索后合并答案：

**原始问题**：「气候变化和 AI 历史有什么相同点？」

**自动拆分**：① 什么是气候变化？ ② AI 的历史是什么？ → 合并两个答案进行对比


In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata

# 为每个文档主题创建专门的查询引擎
tools = [
    QueryEngineTool(
        query_engine=vector_index.as_query_engine(similarity_top_k=1),
        metadata=ToolMetadata(name='knowledge_base', description='Contains facts about climate change, AI history, and health')
    )
]

sub_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)

response = sub_engine.query('Compare the timeline of AI development with climate change milestones.')
print('综合回答:')
print(response)


## 第六章：多文档问答实战

按文档来源分别建索引，再组合查询：

In [ ]:
from llama_index.core import VectorStoreIndex

# 按主题分别构建索引
topic_indices = {}
for doc in documents:
    fname = doc.metadata.get('file_name', 'unknown')
    topic = fname.replace('.txt', '')
    topic_indices[topic] = VectorStoreIndex.from_documents([doc])

print(f'已构建 {len(topic_indices)} 个主题索引: {list(topic_indices.keys())}')

# 跨文档综合问答
questions = [
    ('climate', 'What is the Paris Agreement target?'),
    ('ai_history', 'When did deep learning revolution start?'),
    ('health', 'How much does exercise reduce cardiovascular risk?'),
]

for topic, q in questions:
    qe = topic_indices[topic].as_query_engine()
    ans = qe.query(q)
    print(f'[{topic}] Q: {q}')
    print(f'        A: {ans}\n')


## 小结：何时选 LlamaIndex？

```mermaid
flowchart TD
    START[需要把文档接入 LLM？] -->|是| Q1{文档量和复杂度？}
    Q1 -->|少量文档，简单问答| A1[直接用 API + 检索]
    Q1 -->|多文档，复杂问题| A2[LlamaIndex]
    Q1 -->|需要 Agent 决策| A3[LangChain 或 组合使用]
    A2 --> Q2{问题类型？}
    Q2 -->|精确检索| B1[VectorStoreIndex]
    Q2 -->|全文摘要| B2[SummaryIndex]
    Q2 -->|复合问题| B3[SubQuestionQueryEngine]
```

**LlamaIndex 的核心价值：**
- 文档加载器覆盖 40+ 格式（PDF/网页/数据库/API）
- 智能切块策略（语义切块、层级切块）优于固定长度
- 子问题分解让复杂查询自动化
- 与 LangChain、AutoGen 均可集成

**下一章 →** `03_autogen_multiagent.ipynb`：当单个 LLM 不够用时，如何用多 Agent 协作解决复杂任务。
